# 🐄 Boeuf Tracker — Google Colab

Lance l'UI complète du **Boeuf Tracker** (YOLOv11 + DINOv2 + Re-ID) sur le GPU gratuit de Colab.

**Pourquoi Colab ?**
- 🎮 **GPU T4 (16 GB)** ou **A100 (40 GB)** — bien plus de VRAM qu'une GTX 1660 Ti (6 GB)
- 🧠 Modèles "lourds" supportés : `yolo11l-seg` + `dinov2-base`
- 🌐 UI accessible depuis n'importe quel navigateur via un tunnel **Cloudflare** (gratuit, sans compte)
- 📊 **Moniteur live** intégré : voit en temps réel si la vidéo est traitée
- 🎬 **Auto-détection** des vidéos du repo (la plus petite est utilisée au démarrage)

**Prérequis**
1. Pousse ton repo sur GitHub (privé ou public)
2. Édite la cellule suivante : mets l'URL HTTPS dans `GIT_URL`
3. `Runtime` → `Change runtime type` → **T4 GPU** (ou A100)
4. Exécute les cellules dans l'ordre

À la fin, tu auras :
- Une URL publique `https://xxx.trycloudflare.com` pour l'UI
- Un **dashboard live** qui affiche FPS, frames, source, events, animaux visibles
- Le **log Flask** en temps réel (pour debug du switch de source)
- Le serveur démarre **automatiquement** sur la plus petite vidéo du repo (pas de webcam en Colab)

In [ ]:
# =================================================================
# ⚙️  CONFIGURATION — modifie selon tes besoins
# =================================================================

# URL HTTPS de ton repo Git (obligatoire)
GIT_URL    = "https://github.com/USER/boeuf-tracker.git"   # <-- MODIFIE ICI
GIT_BRANCH = "main"
GIT_TOKEN  = ""        # PAT GitHub pour repo privé (optionnel)

# --- Source vidéo par défaut ---
# "auto"               → la plus petite .mp4 trouvée dans le repo ✅ recommandé en Colab
# "0"                  → webcam (NE MARCHE PAS dans Colab)
# "/path/to/video.mp4" → fichier spécifique (chemin absolu)
VIDEO_SOURCE = "auto"

# --- Modèles (plus gros = +précis mais +VRAM) ---
# T4 (16GB)  : yolo11l-seg + dinov2-base   ✅ confortable
# A100(40GB) : yolo11x-seg + dinov2-large  ✅ très précis
YOLO_MODEL  = "yolo11l-seg.pt"
DINO_MODEL  = "facebook/dinov2-base"

# --- Paramètres de détection ---
THRESHOLD   = 0.65     # Seuil cosine Re-ID (0.4 permissif → 0.8 strict)
CONF        = 0.4      # Confiance min YOLO (0.3 sensible → 0.6 strict)
IMGSZ       = 640      # Résolution YOLO (320 rapide → 1280 précis)
EMBED_EVERY = 10       # Re-embed tous les N frames (perf vs précision)

# --- Réseau ---
PORT = 5000

# --- Injection du token si repo privé ---
_repo_display = GIT_URL
if GIT_TOKEN and "@" not in GIT_URL and "github.com" in GIT_URL:
    GIT_URL = GIT_URL.replace("https://", f"https://x-access-token:{GIT_TOKEN}@")
    _repo_display = GIT_URL.replace(f"x-access-token:{GIT_TOKEN}@", "")

print(f"📦 Repo        : {_repo_display.replace('https://', '')}")
print(f"🎬 Source      : {VIDEO_SOURCE}")
print(f"🤖 YOLO        : {YOLO_MODEL}")
print(f"🧠 DINOv2      : {DINO_MODEL}")
print(f"🔌 Port        : {PORT}")
print(f"⚙️  Settings    : thr={THRESHOLD}  conf={CONF}  imgsz={IMGSZ}  embed_every={EMBED_EVERY}")

In [ ]:
import subprocess, sys

print("⏳ Installation des paquets système (ffmpeg)...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

print("⏳ Installation des dépendances Python...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics>=8.0.0",
    "torch>=2.0.0",
    "torchvision>=0.15.0",
    "transformers>=4.35.0",
    "opencv-python>=4.8.0",
    "numpy>=1.24.0",
    "Pillow>=10.0.0",
    "flask>=3.0.0",
    "werkzeug",
], check=True)

print("✅ Dépendances installées")

In [ ]:
import os, subprocess
from pathlib import Path

if "USER" in GIT_URL or not GIT_URL.strip():
    raise SystemExit(
        "❌ Configure GIT_URL dans la cellule précédente !\n"
        "   Exemple: GIT_URL = 'https://github.com/ton-user/boeuf-tracker.git'"
    )

repo_name = GIT_URL.rstrip("/").split("/")[-1].replace(".git", "")
if "@" in repo_name:
    repo_name = repo_name.split(":")[-1].split("/")[-1]

repo_path = Path("/content") / repo_name

if not repo_path.exists():
    print(f"⏳ Clonage du repo (branche {GIT_BRANCH})...")
    subprocess.run([
        "git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, str(repo_path)
    ], check=True)
    print("✅ Clone OK")
else:
    print(f"✅ Repo déjà présent: {repo_path}")
    print("   (supprime-le pour forcer un re-clone)")

os.chdir(repo_path)
print(f"📂 CWD = {os.getcwd()}")

# Liste les vidéos déjà présentes dans le repo
video_exts = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v"}
videos = []
for entry in sorted(os.listdir(".")):
    full = os.path.join(".", entry)
    if os.path.isfile(full) and os.path.splitext(entry)[1].lower() in video_exts:
        videos.append((entry, os.path.getsize(full)))
if os.path.isdir("uploads"):
    for entry in sorted(os.listdir("uploads")):
        full = os.path.join("uploads", entry)
        if os.path.isfile(full) and os.path.splitext(entry)[1].lower() in video_exts:
            videos.append((full, os.path.getsize(full)))

if videos:
    print(f"\n🎬 {len(videos)} vidéo(s) détectée(s) dans le repo :")
    for name, size in sorted(videos, key=lambda x: x[1]):
        print(f"   • {name:50s}  ({size/1024/1024:6.1f} MB)")
else:
    print("\n⚠️ Aucune vidéo dans le repo. Upload via l'UI ou colle un SAMPLE_URL en cell 8.")

In [ ]:
import torch, subprocess
from pathlib import Path

if not torch.cuda.is_available():
    raise SystemExit(
        "❌ GPU non disponible !\n"
        "   Va dans Runtime → Change runtime type → Hardware accelerator → T4 GPU\n"
        "   Puis ré-exécute cette cellule."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"✅ GPU : {gpu_name}")
print(f"   VRAM: {vram_gb:.1f} GB")

if "T4" in gpu_name.upper():
    if "l-seg" in YOLO_MODEL or "large" in DINO_MODEL.lower():
        print("⚠️  T4 détecté. Si OOM, baisse vers yolo11s-seg + dinov2-small dans la cellule de config.")

yolo_path = Path(YOLO_MODEL)
if not yolo_path.exists():
    print(f"⏳ Téléchargement de {YOLO_MODEL} (~50-100 MB)...")
    tag = "v8.3.0"
    url = f"https://github.com/ultralytics/assets/releases/download/{tag}/{YOLO_MODEL}"
    subprocess.run(["wget", "-q", "--show-progress", url], check=True)
    print("✅ Poids YOLO téléchargés")

print("\n📦 Modèles .pt dans le dossier :")
for p in sorted(Path(".").glob("*.pt")):
    print(f"   - {p.name:25s}  ({p.stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
import os, subprocess, time, urllib.request, urllib.error
from pathlib import Path

# === Auto-détection de la source vidéo ===
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v"}

def find_videos():
    """Liste (chemin, taille) de toutes les vidéos dans le projet."""
    found = []
    cwd = Path(".").resolve()
    for entry in sorted(os.listdir(".")):
        full = cwd / entry
        if full.is_file() and full.suffix.lower() in VIDEO_EXTS:
            found.append((str(full), full.stat().st_size))
    uploads = cwd / "uploads"
    if uploads.is_dir():
        for entry in sorted(os.listdir(uploads)):
            full = uploads / entry
            if full.is_file() and full.suffix.lower() in VIDEO_EXTS:
                found.append((str(full), full.stat().st_size))
    return found

if VIDEO_SOURCE == "auto":
    candidates = find_videos()
    if not candidates:
        raise SystemExit(
            "❌ Aucune vidéo trouvée dans le repo !\\n"
            "   → Mets une .mp4 dans le repo, OU\\n"
            "   → Change VIDEO_SOURCE = '0' (essaiera webcam, échec en Colab), OU\\n"
            "   → Upload une vidéo via l'UI après démarrage (voir cell 7)"
        )
    # Prend la plus PETITE (téléchargement + test rapides)
    candidates.sort(key=lambda x: x[1])
    INITIAL_SOURCE = candidates[0][0]
    print(f"🎬 AUTO-PICK : {INITIAL_SOURCE}")
    print(f"   ({candidates[0][1]/1024/1024:.1f} MB — {len(candidates)} vidéo(s) dispo dans le repo)")
elif VIDEO_SOURCE == "0":
    INITIAL_SOURCE = "0"
    print("🎬 Source = webcam 0 (⚠️ ne marche pas dans Colab)")
elif os.path.exists(VIDEO_SOURCE):
    INITIAL_SOURCE = VIDEO_SOURCE
    print(f"🎬 Source forcée : {INITIAL_SOURCE}")
else:
    raise SystemExit(f"❌ VIDEO_SOURCE invalide : {VIDEO_SOURCE!r} (fichier introuvable)")

# === Nettoyage runs précédents ===
subprocess.run(["pkill", "-f", "python app.py"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "cloudflared"],    stderr=subprocess.DEVNULL)
time.sleep(2)

# === Démarrage Flask ===
flask_cmd = [
    "python", "app.py",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--source", INITIAL_SOURCE,
    "--device", "cuda:0",
    "--yolo-model", YOLO_MODEL,
    "--dino-model", DINO_MODEL,
    "--threshold", str(THRESHOLD),
    "--conf",      str(CONF),
    "--imgsz",     str(IMGSZ),
    "--embed-every", str(EMBED_EVERY),
]

log_path = Path("/tmp/flask.log")
log_path.unlink(missing_ok=True)
log_f = open(log_path, "wb", 0)

print()
print("⏳ Démarrage du serveur Flask...")
print("   (chargement YOLO + DINOv2 + DB → ~30-60s)")
print()

flask_proc = subprocess.Popen(
    flask_cmd,
    stdout=log_f,
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)

ready = False
for i in range(120):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/stats", timeout=1).read()
        ready = True
        break
    except (urllib.error.URLError, ConnectionResetError):
        if flask_proc.poll() is not None:
            print("❌ Le serveur a crashé. Dernières lignes du log :")
            subprocess.run(["tail", "-40", "/tmp/flask.log"])
            raise SystemExit(1)
        if i % 5 == 0:
            print(f"   ... chargement ({i*2}s)")

if not ready:
    print("❌ Timeout (4 min).")
    subprocess.run(["tail", "-60", "/tmp/flask.log"])
    raise SystemExit(1)

print(f"✅ Serveur Flask prêt sur http://127.0.0.1:{PORT}")
print(f"   PID Flask = {flask_proc.pid}")

import json as _j
try:
    stats = _j.loads(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/stats").read())
    print(f"   Device actif : {stats.get('device')}")
    print(f"   YOLO chargé  : {stats.get('current', {}).get('yolo_model')}")
    print(f"   Source       : {stats.get('source_label')}")
    print(f"   Path         : {stats.get('current_source_path')}")
except Exception as e:
    print(f"   (stats non lisibles: {e})")

In [ ]:
import subprocess, re, time, os, urllib.request, json
from pathlib import Path
from IPython.display import clear_output

# === Installation cloudflared ===
if not Path("/usr/local/bin/cloudflared").exists() and not Path("/usr/bin/cloudflared").exists():
    print("⏳ Installation de cloudflared...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        "-O", "/tmp/cloudflared.deb",
    ], check=True)
    subprocess.run(["dpkg", "-i", "/tmp/cloudflared.deb"], check=True)

# === Démarrage tunnel ===
tunnel_log = Path("/tmp/tunnel.log")
tunnel_log.unlink(missing_ok=True)
tunnel_f = open(tunnel_log, "wb", 0)

print("⏳ Création du tunnel Cloudflare...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=tunnel_f,
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)

url = None
for i in range(90):
    time.sleep(2)
    content = tunnel_log.read_text(errors="ignore")
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', content)
    if m:
        url = m.group(1)
        break
    if tunnel_proc.poll() is not None:
        print("❌ Tunnel crashé. Log :")
        print(content[-1500:])
        raise SystemExit(1)

if not url:
    print("❌ Pas d'URL publique après 3 min.")
    print(tunnel_log.read_text(errors="ignore")[-2000:])
    raise SystemExit(1)

print()
print("=" * 70)
print(f"  🌐  UI ACCESSIBLE À :")
print(f"      {url}")
print("=" * 70)
print()
print("📊 Démarrage du MONITEUR LIVE (rafraîchissement toutes les 3s)...")
print("   → Upload une vidéo via l'UI et regarde le moniteur")
print("   → Ctrl+C pour ARRÊTER le monitoring (serveur + tunnel restent UP)")
print()


# === Helpers pour le moniteur ===

def get_stats():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/stats", timeout=2) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"_error": str(e)}


def get_videos():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/videos", timeout=2) as r:
            return json.loads(r.read()).get("videos", [])
    except Exception:
        return []


def get_diag():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/diag", timeout=2) as r:
            return json.loads(r.read())
    except Exception:
        return {}


def tail_log(n=12):
    try:
        with open("/tmp/flask.log", "rb") as f:
            f.seek(0, 2)
            size = f.tell()
            f.seek(max(0, size - 8192))
            data = f.read().decode(errors="ignore")
            return data.splitlines()[-n:]
    except Exception:
        return []


def send_source(path):
    try:
        req = urllib.request.Request(
            f"http://127.0.0.1:{PORT}/api/source/file",
            data=json.dumps({"path": path}).encode(),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=5) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"ok": False, "error": str(e)}


# === Boucle du moniteur live ===

prev_fc = -1
prev_src = None

try:
    while True:
        clear_output(wait=True)

        # BANNIÈRE URL
        print("=" * 70)
        print(f"  🌐  UI LIVE : {url}")
        print("=" * 70)

        stats = get_stats()
        diag  = get_diag()

        if "_error" in stats:
            print(f"\n❌ /api/stats inaccessible : {stats['_error']}")
        else:
            fps     = stats.get("fps", 0)
            fc      = stats.get("frame_count", 0)
            src     = stats.get("source_label", "?")
            dev     = stats.get("device", "?")
            events  = stats.get("events", [])
            cur     = stats.get("current", {})
            active  = stats.get("active", [])
            behavior = stats.get("behavior", [])

            if fc == 0 and not events:
                status = "🟡 INITIALISATION — aucune frame traitée"
            elif fc == 0 and events:
                status = "🟠 0 FRAME — events loggés mais pas de frame"
            elif fc > prev_fc:
                status = f"🟢 TRAITEMENT OK (Δ +{fc - prev_fc})"
            elif fc == prev_fc and fc > 0:
                status = "🟠 STAGNANT — frames ne progressent plus"
            else:
                status = "🔴 BLOQUÉ"

            src_changed = "  🔄 SOURCE CHANGÉE !" if src != prev_src else ""

            print(f"\n{status}{src_changed}")
            print(f"   📂 Source      : {src}")
            print(f"   🎮 Device      : {dev}")
            print(f"   🎬 FPS         : {fps:.1f}")
            print(f"   🖼️  Frames      : {fc}")
            print(f"   🤖 YOLO        : {cur.get('yolo_model', '?')}")
            print(f"   📐 imgsz       : {cur.get('imgsz', '?')}")
            print(f"   🎚️  threshold   : {cur.get('threshold', '?')}")
            print(f"   🎯 conf        : {cur.get('conf', '?')}")

            if events:
                print(f"\n📝 Derniers événements (du + récent au + ancien) :")
                for e in events[:10]:
                    print(f"   • {e}")
            else:
                print(f"\n📝 Aucun événement pour l'instant")
                print(f"   💡 Upload une vidéo via l'UI → elle apparaîtra ici en 'NEW' ou 'MATCH'")

            if active:
                print(f"\n🐄 Animaux visibles ({len(active)}) :")
                for a in active[:8]:
                    print(f"   • {a.get('name', '?'):15s}  conf={a.get('conf', 0):.2f}  track_id={a.get('track_id', '?')}")
            else:
                print(f"\n🐄 Aucun animal visible actuellement")

            if behavior:
                print(f"\n🏃 Comportements :")
                for b in behavior[:5]:
                    print(f"   • {b.get('name', '?'):15s} → {b.get('action', '?')} (vitesse={b.get('speed', 0):.1f})")

        # DIAGNOSTIC
        desired_src = diag.get("state", {}).get("desired_source", None)
        cur_src     = diag.get("state", {}).get("current_source_path", None)
        if desired_src:
            print(f"\n🔄 Switch demandé vers : {desired_src}")
        if cur_src and cur_src != 0:
            print(f"✅ Source actuelle (interne) : {cur_src}")

        # VIDÉOS DISPONIBLES
        videos = get_videos()
        if videos:
            print(f"\n🎬 Vidéos détectées par l'app ({len(videos)}) :")
            for v in videos[:5]:
                print(f"   • [{v.get('source', '?'):8s}] {v.get('name', '?')}  ({v.get('size_mb', 0):.1f} MB)")

        # LOG FLASK
        log_lines = tail_log(12)
        if log_lines:
            print(f"\n📜 Log Flask (12 dernières lignes — couleurs perdues) :")
            for line in log_lines:
                if line.strip():
                    print(f"   {line[:200]}")

        print(f"\n" + "─" * 70)
        print(f"💡 TESTS RAPIDES (à exécuter dans une autre cellule si besoin) :")
        print(f"   # Lister les vidéos que l'app voit :")
        print(f"   get_videos()")
        print(f"   # Forcer un switch de source :")
        print(f"   send_source('/content/boeuf-tracker/107414-678258609_medium.mp4')")
        print(f"   # Stats brutes :")
        print(f"   get_stats()")
        print("─" * 70)
        print(f"⏳ Prochain refresh dans 3s ... (Ctrl+C pour stopper le monitoring)")

        prev_fc  = fc if 'fc' in dir() else 0
        prev_src = src if 'src' in dir() else None

        time.sleep(3)

except KeyboardInterrupt:
    clear_output(wait=True)
    print()
    print("=" * 70)
    print("🛑  MONITEUR ARRÊTÉ")
    print("=" * 70)
    print(f"✅ Serveur Flask    : toujours UP")
    print(f"✅ Tunnel Cloudflare : toujours UP")
    print(f"🌐 URL              : {url}")
    print()
    print("📌 Pour relancer le monitoring : ré-exécute cette cellule.")
    print("📌 Pour switcher de source manuellement :")
    print(f"   send_source('/content/boeuf-tracker/107414-678258609_medium.mp4')")
    print()
    print("=" * 70)

In [ ]:
# OPTIONNEL: télécharge une vidéo d'exemple pour tester rapidement.
# Sinon, utilise les vidéos déjà dans le repo (auto-détectées par cell 6).

SAMPLE_URL = ""   # ← colle ici une URL directe vers un .mp4 de bovins
if SAMPLE_URL:
    print(f"⏳ Téléchargement de la vidéo d'exemple...")
    subprocess.run(["wget", "-q", "--show-progress", SAMPLE_URL, "-O", "sample_cattle.mp4"], check=True)
    size_mb = Path("sample_cattle.mp4").stat().st_size / 1024 / 1024
    print(f"✅ Vidéo téléchargée: sample_cattle.mp4 ({size_mb:.1f} MB)")
    print(f"   → Redémarre le serveur (ré-exécute cell 6) pour la prendre en compte.")
else:
    print("💡 Pas de SAMPLE_URL configuré.")
    print("   → Le serveur tourne déjà sur une vidéo auto-détectée du repo.")
    print("   → Upload d'autres vidéos via le bouton 'Upload' dans l'UI.")
    print("   → Ou mets une URL directe ci-dessus + redémarre le serveur.")

## 🎉 C'est en ligne !

Ouvre l'URL affichée par le moniteur dans ton navigateur. Tu devrais voir :
- Le **flux MJPEG** en temps réel avec les silhouettes des bovins
- Les **sliders** pour ajuster seuil Re-ID, confiance, imgsz à chaud
- Le **journal** des événements (`NEW`, `MATCH`, `LOOP`, etc.)
- La **liste des animaux** détectés avec leur couleur stable

### 🎬 Vidéos utilisées

Le serveur **démarre automatiquement** sur la plus petite `.mp4` trouvée dans le repo.
Tu peux switcher à tout moment via :
- L'UI → dropdown **"Source"** → choisis une autre vidéo du projet
- L'UI → bouton **"Upload"** → upload depuis ton navigateur
- Le notebook → `send_source("/content/boeuf-tracker/NOM.mp4")` depuis n'importe quelle cellule

### 📺 Pendant que l'UI tourne

Le **moniteur live** (cellule précédente) t'affiche en temps réel :
- 🟢/🟠/🔴 état du processing (frames qui avancent ou bloquées)
- 📝 tous les events que le serveur loggue (`NEW Boeuf_001`, `MATCH Marguerite`, etc.)
- 📜 les 12 dernières lignes du log Flask (erreurs, switches de source, OOM…)
- 🎬 les vidéos détectées par l'app

### 🛑 Pour arrêter le monitoring (sans tuer le serveur)
- Bouton **⏹ Stop** de la cellule, OU Ctrl+C
- Le serveur Flask + le tunnel continuent à tourner

### 🛑 Pour tout arrêter
- Ferme l'onglet Colab, ou `Runtime` → `Manage sessions` → Terminate

### 🔧 Troubleshooting

| Symptôme dans le moniteur | Cause probable | Solution |
|---|---|---|
| `🟡 INITIALISATION` + log vide | YOLO charge encore | Attendre 30-60s |
| `🟠 STAGNANT` + `src=...mp4` | Vidéo finie ou bloquée | Change de source dans l'UI |
| `🔄 SOURCE CHANGÉE !` mais reste sur l'ancienne | Fichier vidéo illisible | Vérifie le codec (H.264 conseillé) |
| Events `NEW` mais aucun `MATCH` | DB vide (normal au 1er run) | Normal ! Bovins nommés Boeuf_001… |
| Log montre `OOM CUDA` | VRAM insuffisante | Baisse `YOLO_MODEL` à `yolo11s-seg.pt` |
| Log montre `[Switch] ERREUR ouverture` | Fichier vidéo introuvable | Vérifie le chemin ou utilise Upload UI |